# Policy-Aware RAG Evaluation Analysis

This notebook analyzes the current Function App output, including per-step latency and token estimates captured in the single request-level audit log.

The current app exposes /api/rag and stores all step telemetry under a single request record in AuditStorage, which makes it possible to measure retrieval, policy evaluation, and response guardrail timing individually.


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import answer_relevancy, context_precision, context_recall, faithfulness
    HAS_RAGAS = True
except Exception:
    HAS_RAGAS = False

RESULTS_PATH = os.getenv("EVALUATIONS_RESULTS_PATH", "evaluation_results.json")
print(f"Results file: {RESULTS_PATH}")


In [ ]:
def load_results(path):
    payload = json.loads(Path(path).read_text(encoding='utf-8'))
    return payload.get('results', [])

records = load_results(RESULTS_PATH)
df = pd.DataFrame(records)
display(df[['case_type', 'status_code', 'expected_outcome', 'actual_outcome', 'passed', 'http_latency_ms', 'total_step_latency_ms']].head())


In [ ]:
step_rows = []
for row in df.itertuples():
    for step in getattr(row, 'step_metrics', []):
        step_rows.append({
            'case_type': row.case_type,
            'stepName': step.get('stepName'),
            'executionStatus': step.get('executionStatus'),
            'latency_ms': step.get('latency_ms', 0),
            'query_tokens': step.get('query_tokens', 0),
            'answer_tokens': step.get('answer_tokens', 0),
            'document_count': step.get('document_count'),
        })

steps_df = pd.DataFrame(step_rows)
display(steps_df.groupby('stepName').agg(step_count=('stepName', 'size'), avg_latency_ms=('latency_ms', 'mean')).sort_values('avg_latency_ms', ascending=False))


In [ ]:
summary = df.groupby('case_type').agg(
    total=('question', 'size'),
    passed=('passed', 'sum'),
    pass_rate=('passed', 'mean'),
    avg_http_latency_ms=('http_latency_ms', 'mean'),
    avg_step_latency_ms=('total_step_latency_ms', 'mean'),
).reset_index()
summary['pass_rate'] = summary['pass_rate'].round(3)
summary['avg_http_latency_ms'] = summary['avg_http_latency_ms'].round(1)
summary['avg_step_latency_ms'] = summary['avg_step_latency_ms'].round(1)
display(summary.sort_values('pass_rate'))


In [ ]:
if HAS_RAGAS:
    ragas_rows = []
    for row in df.itertuples():
        answer = str(row.response.get('answer', '') if isinstance(row.response, dict) else '')
        sources = row.response.get('sources', []) if isinstance(row.response, dict) else []
        if not answer:
            continue
        dataset = Dataset.from_dict({
            'question': [row.question],
            'answer': [answer],
            'contexts': [[str(item) for item in sources]],
        })
        scores = evaluate(
            dataset,
            metrics=[answer_relevancy, context_precision, context_recall, faithfulness],
        )
        ragas_rows.append({
            'case_type': row.case_type,
            **{k: float(v) for k, v in scores.to_dict().items() if isinstance(v, (int, float))},
        })
    ragas_df = pd.DataFrame(ragas_rows)
    display(ragas_df.groupby('case_type').mean(numeric_only=True).round(3))
else:
    print('RAGAS is not installed; showing proxy pass-rate summary instead.')
    display(df.groupby('case_type').agg(pass_rate=('passed', 'mean')).round(3))
